# HRS Gold Layer DDL Functional Specificationanalytic_

## BMI Race and Gender Statistics

---

# 1. Document Information

| Property           | Value                                                                    |
| ------------------ | ------------------------------------------------------------------------ |
| Document Name      | HRS Gold Layer DDL Functional Specification — BMI Race Gender Statistics |
| Version            | 1.0                                                                      |
| Author             | Perez                                                                    |
| AI Assistant       | ChatGPT and Claude                                                       |
| Last Updated       | 2026-09-16                                                               |
| Target Platform    | Databricks                                                               |
| Compute            | Serverless                                                               |
| Runtime            | client.5.12                                                              |
| SQL Dialect        | Databricks SQL / Spark SQL                                               |
| Storage Format     | Delta                                                                    |
| Specification Type | DDL Only                                                                 |
| Target Data Layer  | Gold                                                                     |
| Primary Purpose    | Curated analytical and visualization data                                |

---

# 2. Purpose

This specification defines the physical structure of the Gold-layer analytical table used to store descriptive BMI statistics by HRS cohort, survey wave, race, and gender.

The table is designed to support:

* Data analytics
* Data visualization
* Statistical analysis
* Reporting
* Public health analysis

The table stores aggregated BMI statistics rather than individual BMI observations.

### Important Design Decision

The `bmi` value itself is **not stored as a target dimension**.

The target table must **not** contain a separate row for every unique BMI value.

Instead, BMI statistics are grouped by:

```text
Cohort
+
Wave
+
Race
+
Gender
```

The resulting table contains descriptive statistics for each combination of these dimensions.

---

# 3. Gold Layer Role

The table belongs to the HRS Gold analytical layer.

```text
RAND HRS Dataset
       │
       ▼
Bronze Layer
Raw / Source Data
       │
       ▼
Silver CDM
Standardized Relational Data
       │
       ▼
Gold Layer
Curated Analytical Data
       │
       ▼
BMI Race/Gender Statistics
       │
       ├───────────────┐
       ▼               ▼
   Analytics       Visualization
```

The Gold table is an analytical table and does not reproduce the underlying respondent-level Silver CDM structure.

---

# 4. Analytical Use Case

| Property          | Value                                                                    |
| ----------------- | ------------------------------------------------------------------------ |
| Use Case ID       | UC_DA_001                                                                |
| Use Case Name     | HRS BMI Race and Gender Statistics                                       |
| Business Question | How do BMI statistics vary by HRS cohort, survey wave, race, and gender? |
| Primary Users     | Data Analysts / Researchers / Policy Analysts                            |
| Intended Use      | Analytics / Visualization / Reporting                                    |

## 4.1 Business Question

> How do BMI descriptive statistics vary across HRS cohorts, survey waves, race groups, and gender groups?

---

# 5. Gold Table Grain

One row represents:

> **One combination of HRS cohort, survey wave, race, and gender, with associated descriptive BMI statistics.**

| Grain Component       | Description                                              |
| --------------------- | -------------------------------------------------------- |
| Cohort                | HRS cohort                                               |
| Wave                  | HRS survey wave                                          |
| Race                  | Respondent race category                                 |
| Gender                | Respondent gender category                               |
| Measures / Statistics | Descriptive BMI statistics for the dimension combination |

### Grain Definition

The analytical grain is:

```text
cohort_id
+
wave_id
+
raracem
+
ragender
```

### Important

The `bmi` value is **not part of the analytical grain**.

The table therefore does not create separate groups for individual BMI values.

---

# 6. Gold Table Type

| Property           | Value                                                              |
| ------------------ | ------------------------------------------------------------------ |
| GOLD_TABLE_TYPE    | Analytical                                                         |
| ANALYTICAL_ROLE    | Aggregate Statistics                                               |
| ANALYTICAL_PURPOSE | Store descriptive BMI statistics by cohort, wave, race, and gender |

---

# 7. Target Table Parameters

| Parameter                | Value                                                                          |
| ------------------------ | ------------------------------------------------------------------------------ |
| TARGET_TABLE_NAME        | `analytic_hrs_bmi_race_gender_stats`                                               |
| TARGET_TABLE_DESCRIPTION | BMI descriptive statistics table grouped by HRS cohort, wave, race, and gender |
| CATALOG_NAME             | `dev_catalog`                                                                  |
| SCHEMA_NAME              | `gld_star_hrs`                                                                 |
| STORAGE_FORMAT           | `DELTA`                                                                        |
| TABLE_TYPE               | `Managed Table`                                                                |

### Fully Qualified Table Name

```text
dev_catalog.gld_star_hrs.analytic_hrs_bmi_race_gender_stats
```

---

# 8. Gold Table Design

The table contains four primary analytical dimensions/identifiers:

* `cohort_id`
* `wave_id`
* `raracem`
* `ragender`

It also contains the following descriptive BMI statistics:

* `bmi_count`
* `bmi_mean`
* `bmi_sd`
* `bmi_min`
* `bmi_max`

The table does not contain a `bmi` column.

---

# 9. Key Strategy

The table uses a system-generated surrogate primary key.

| Property                     | Value                                   |
| ---------------------------- | --------------------------------------- |
| Surrogate Key Required       | Yes                                     |
| Surrogate Key Column         | `analytic_hrs_bmi_race_gender_stats_id`     |
| Surrogate Key Type           | `BIGINT`                                |
| Generated Always As Identity | Yes                                     |
| Primary Key                  | `analytic_hrs_bmi_race_gender_stats_id`     |
| Analytical Key               | `cohort_id, wave_id, raracem, ragender` |
| Business Key                 | `cohort_id, wave_id, raracem, ragender` |

The surrogate key is generated by Databricks.

The analytical/business key identifies the statistical observation.

---

# 10. Dimension Columns

| Column Name | Databricks Type | Nullable | Description                                 |
| ----------- | --------------- | -------- | ------------------------------------------- |
| `cohort_id` | BIGINT          | No       | Foreign key identifying the HRS cohort      |
| `wave_id`   | BIGINT          | No       | Foreign key identifying the HRS survey wave |
| `raracem`   | INT             | Yes      | RAND HRS race category                      |
| `ragender`  | INT             | Yes      | RAND HRS gender category                    |
| `hacohort`  | INT             | Yes      | HRS cohort identifier                       |

### Note

`cohort_id` and `wave_id` are structural foreign keys.

`raracem` and `ragender` are analytical grouping attributes.

`hacohort` is retained as a business attribute in the target table.

---

# 11. Measure Columns

| Column Name | Databricks Type | Nullable | Description                                           |
| ----------- | --------------- | -------- | ----------------------------------------------------- |
| `bmi_count` | INT             | Yes      | Number of records included in the BMI statistics      |
| `bmi_mean`  | DOUBLE          | Yes      | Mean BMI for the analytical grouping                  |
| `bmi_sd`    | DOUBLE          | Yes      | Standard deviation of BMI for the analytical grouping |
| `bmi_min`   | INT             | Yes      | Minimum BMI value for the analytical grouping         |
| `bmi_max`   | INT             | Yes      | Maximum BMI value for the analytical grouping         |

### Important

These columns contain aggregated BMI statistics.

The individual BMI value is not stored in the Gold table.

---

# 12. Derived Analytical Columns

```text
Derived Analytical Columns Required: No
```

No additional derived analytical columns are defined in this specification.

The statistical measures listed in Section 11 are physical target columns.

The formulas and transformation logic used to calculate these measures belong in the corresponding **Gold DML Functional Specification**.

---

# 13. Gold Column Definition Matrix

| Column Name                         | Category           | Databricks Type | Nullable | Description                                      |
| ----------------------------------- | ------------------ | --------------- | -------- | ------------------------------------------------ |
| `analytic_hrs_bmi_race_gender_stats_id` | Key            | BIGINT          | No       | System-generated surrogate key                   |
| `cohort_id`                         | Foreign Key        | BIGINT          | No       | Foreign key to HRS cohort dimension              |
| `wave_id`                           | Foreign Key        | BIGINT          | No       | Foreign key to HRS wave dimension                |
| `raracem`                           | Dimension          | INT             | Yes      | RAND HRS race category                           |
| `ragender`                          | Dimension          | INT             | Yes      | RAND HRS gender category                         |
| `hacohort`                          | Business Attribute | INT             | Yes      | HRS cohort identifier                            |
| `bmi_count`                         | Measure            | INT             | Yes      | Number of records included in the BMI statistics |
| `bmi_mean`                          | Measure            | DOUBLE          | Yes      | Mean BMI                                         |
| `bmi_sd`                            | Measure            | DOUBLE          | Yes      | Standard deviation of BMI                        |
| `bmi_min`                           | Measure            | INT             | Yes      | Minimum BMI                                      |
| `bmi_max`                           | Measure            | INT             | Yes      | Maximum BMI                                      |
| `create_date`                       | Audit              | DATE            | No       | Record creation date                             |
| `update_date`                       | Audit              | DATE            | No       | Last update date                                 |
| `active`                            | Audit              | BOOLEAN         | No       | Active indicator                                 |

---

# 14. Parent Table Dependencies

The Gold table has two parent-table dependencies.

| Parent Table                              | Purpose                             |
| ----------------------------------------- | ----------------------------------- |
| `dev_catalog.slv_cdm_hrs.dim_cohort` | Provides HRS cohort identifier      |
| `dev_catalog.slv_cdm_hrs.dim_wave`   | Provides HRS survey wave identifier |

---

# 15. Foreign Key Relationships

| Child Column | Parent Table                              | Parent Column |
| ------------ | ----------------------------------------- | ------------- |
| `cohort_id`  | `dev_catalog.slv_cdm_hrs.dim_cohort` | `cohort_id`   |
| `wave_id`    | `dev_catalog.slv_cdm_hrs.dim_wave`   | `wave_id`     |

### Relationship Diagram

```text
analytic_hrs_bmi_race_gender_stats
          │
          ├── cohort_id ──────► dim_cohort.cohort_id
          │
          └── wave_id ────────► dim_wave.wave_id
```

---

# 16. Table Constraints

| Constraint Type | Required |
| --------------- | -------- |
| PRIMARY KEY     | Yes      |
| FOREIGN KEY     | Yes      |
| UNIQUE          | Yes      |
| NOT NULL        | Yes      |
| CHECK           | No       |
| IDENTITY        | Yes      |

### Primary Key

```text
analytic_hrs_bmi_race_gender_stats_id
```

### Business Key

```text
cohort_id
+
wave_id
+
raracem
+
ragender
```

The business key represents the analytical grouping defined by the table grain.

---

# 17. Audit Columns

The standard HRS Gold audit columns are required.

| Column        | Type    | Nullable | Description          |
| ------------- | ------- | -------- | -------------------- |
| `create_date` | DATE    | No       | Record creation date |
| `update_date` | DATE    | No       | Last update date     |
| `active`      | BOOLEAN | No       | Active indicator     |

---

# 18. Table Comment

The Gold table comment should describe the analytical purpose:

> BMI descriptive statistics table grouped by HRS cohort, wave, race, and gender.

---

# 19. Column Comments

The generated DDL should include comments for all physical columns.

Comments must describe the business purpose of each column.

At minimum, comments are required for:

* Surrogate primary key
* Foreign keys
* Business attributes
* Analytical dimensions
* BMI measures
* Audit columns

---

# 20. DROP TABLE Requirement

| Requirement          | Value |
| -------------------- | ----- |
| DROP TABLE IF EXISTS | Yes   |

The generated development DDL should include:

```sql
DROP TABLE IF EXISTS
```

before creating the table.

---

# 21. Physical Table Requirements

The generated table must:

* Use the `dev_catalog` catalog.
* Use the `gld_star_hrs` schema.
* Use the specified table name.
* Be a Managed Table.
* Use Delta storage.
* Use explicit column definitions.
* Use the specified data types.
* Apply the specified nullability.
* Create the identity primary key.
* Create the specified foreign keys.
* Create the business-key uniqueness constraint.
* Include table comments.
* Include column comments.

---

# 22. SQL Generation Requirements

The generated SQL must:

1. Use Databricks SQL / Spark SQL.
2. Target Serverless compute.
3. Use runtime `client.5.12`.
4. Use Delta storage.
5. Create a Managed Table.
6. Use uppercase SQL keywords.
7. Use consistent indentation.
8. Use explicit column definitions.
9. Create the system-generated identity column.
10. Create the primary key constraint.
11. Create the foreign-key constraints.
12. Create the business-key uniqueness constraint.
13. Include table and column comments.
14. Use only DDL statements.

---

# 23. DDL-Only Requirement

The generated SQL must contain DDL only.

The DDL may include:

```sql
DROP TABLE IF EXISTS
CREATE TABLE
```

The generated DDL must not contain:

```text
INSERT
UPDATE
DELETE
MERGE
SELECT
```

No data transformation or loading logic should be generated.

---

# 24. Unsupported Objects

The DDL generator must not create:

```text
INSERT
UPDATE
DELETE
MERGE
VIEWS
STORED PROCEDURES
FUNCTIONS
INDEXES
PARTITIONS
ZORDER
OPTIMIZE
```

These activities are outside the responsibility of the Gold DDL specification.

---

# 25. DDL / DML Separation

The DDL specification defines the physical Gold table.

The DML specification will subsequently define how the table is populated.

```text
Analytical Use Case
        │
        ▼
Gold DDL Specification
        │
        ▼
Generate DDL
        │
        ▼
Create Gold Delta Table
        │
        ▼
Validate Structure
        │
        ▼
Gold DML Specification
        │
        ▼
Calculate BMI Statistics
        │
        ▼
Load Gold Table
```

### DDL

> What does the table look like?

### DML

> How is the table populated?

---

# 26. Structural Validation Requirements

After the DDL is executed, the following should be validated.

| Validation          | Required |
| ------------------- | -------- |
| Table Exists        | Yes      |
| Correct Catalog     | Yes      |
| Correct Schema      | Yes      |
| Correct Table Name  | Yes      |
| Delta Format        | Yes      |
| Managed Table       | Yes      |
| Expected Columns    | Yes      |
| Correct Data Types  | Yes      |
| Correct Nullability | Yes      |
| Identity Column     | Yes      |
| Primary Key         | Yes      |
| Cohort Foreign Key  | Yes      |
| Wave Foreign Key    | Yes      |
| Business Key        | Yes      |
| Audit Columns       | Yes      |

### Analytical Grain Validation

The table structure must support the following grain:

```text
cohort_id
+
wave_id
+
raracem
+
ragender
```

The `bmi` column must not exist in the target table.

---

# 27. DDL Workflow

```text
Analytical Use Case
        │
        ▼
Define Business Question
        │
        ▼
Define Analytical Grain
        │
        ▼
Define Dimensions
        │
        ▼
Define BMI Measures
        │
        ▼
Define Gold Columns
        │
        ▼
Define Keys and Constraints
        │
        ▼
Complete DDL Specification
        │
        ▼
Generate DDL
        │
        ▼
Review Generated SQL
        │
        ▼
Execute in Databricks
        │
        ▼
Validate Gold Table
        │
        ▼
Begin Gold DML Specification
```

---

# 28. DDL Deliverable

The generated SQL file should follow the repository convention:

```text
/sql/ddl/create_<TARGET_TABLE_NAME>.sql
```

For this table:

```text
/sql/ddl/create_analytic_hrs_bmi_race_gender_stats.sql
```

The final DDL deliverable must contain:

```text
SQL ONLY
```

---

# 29. AI DDL Generation Instructions

Generate a complete Databricks SQL DDL script from this specification.

The generated script must:

1. Follow this specification exactly.
2. Create the specified Gold table.
3. Use `dev_catalog.gld_star_hrs`.
4. Use table name `analytic_hrs_bmi_race_gender_stats`.
5. Create only the columns defined in the Column Definition Matrix.
6. Create `analytic_hrs_bmi_race_gender_stats_id` as a `BIGINT GENERATED ALWAYS AS IDENTITY`.
7. Define the surrogate primary key.
8. Define the `cohort_id` foreign key.
9. Define the `wave_id` foreign key.
10. Define the business-key uniqueness constraint where specified.
11. Include the audit columns.
12. Include table and column comments.
13. Use Delta storage.
14. Create a Managed Table.
15. Include `DROP TABLE IF EXISTS`.
16. Use DDL statements only.
17. Do not create a `bmi` column.
18. Do not create a separate grouping level for individual BMI values.
19. Do not generate INSERT, UPDATE, DELETE, or MERGE statements.
20. Do not generate transformation logic.
21. Do not invent columns.
22. Do not invent constraints.
23. Do not invent business rules.
24. Return SQL only.

---

# 30. Final Specification Checklist

* [x] Analytical Use Case defined
* [x] Business Question defined
* [x] Gold Table Grain defined
* [x] Gold Table Type defined
* [x] Target Table Name defined
* [x] Target Catalog defined
* [x] Target Schema defined
* [x] Delta Storage confirmed
* [x] Managed Table confirmed
* [x] Key Strategy defined
* [x] Dimension Columns defined
* [x] Measure Columns defined
* [x] Derived Columns addressed
* [x] Gold Column Definition Matrix completed
* [x] Parent Table Dependencies defined
* [x] Foreign Key Relationships defined
* [x] Constraints defined
* [x] Audit Columns defined
* [x] Table Comment defined
* [x] Column Comments defined
* [x] DROP TABLE requirement defined
* [x] Structural Validation Requirements defined
* [x] Analytical Grain Validation defined

---

# 31. What Does NOT Belong in This Specification?

The following information belongs in the corresponding **Gold DML Functional Specification**:

* Silver source tables
* RAND HRS source variables
* Source-to-target mappings
* Source data transformations
* BMI calculation logic
* Filtering rules
* Aggregation formulas
* Lookup logic
* Data-quality transformations
* INSERT logic
* MERGE logic
* Load sequencing

The DDL specification defines the physical structure only.

---

# 32. Gold DDL → Gold DML Handoff

The DDL specification is complete when the physical Gold table has been created and structurally validated.

The DML specification begins after the table structure has been approved.

```text
              GOLD DDL
                 │
                 ▼
          Physical Table
                 │
                 ▼
         DDL Validation
                 │
                 ▼
          APPROVED TABLE
                 │
                 ▼
              GOLD DML
                 │
                 ▼
        Source + Mapping
                 │
                 ▼
          Transformations
                 │
                 ▼
               Load
```

The Gold DML specification may reference this approved table structure rather than redefining it.

---

# 33. Architectural Principle

The HRS Gold Layer follows this sequence:

> **Define the analytical requirement first, define the physical table second, and define the data-loading process third.**

For this use case:

```text
BMI Analytical Requirement
          │
          ▼
Cohort × Wave × Race × Gender
          │
          ▼
Gold DDL
          │
          ▼
analytic_hrs_bmi_race_gender_stats
          │
          ▼
Gold DML
          │
          ▼
BMI Descriptive Statistics
```

The resulting Gold table is designed for analytical consumption without storing individual BMI values as a grouping dimension.
